# Pequeño estudio de diabetes

Datos de progresión de la diabetes a un año de seguimiento (Efron, Hastie, Johnstone y
Tibshirani, *Least Angle Regression*, 2004). 442 pacientes, 10 variables basales y una
respuesta cuantitativa `Y`.

## Objetivos

1. Calcular los valores medios y la varianza para todos los valores.
2. Graficar diagramas de caja (boxplots) para IMC, PA y Y dependiendo del género.
3. ¿Cuál es la distribución de las variables Edad, Sexo, IMC y Y?
4. Probar la correlación entre diferentes variables y la progresión de la enfermedad (Y).
5. Probar la hipótesis de que el grado de progresión de la diabetes es diferente entre
   hombres y mujeres.

## Origen de los datos

Fuente: <https://www4.stat.ncsu.edu/~boos/var.select/diabetes.tab.txt>

Ojo con las dos versiones del archivo que circulan en esa página:

| Archivo | Nombres de columna |
| --- | --- |
| `diabetes.tab.txt` (el que usamos) | `AGE SEX BMI BP S1 S2 S3 S4 S5 S6 Y` |
| `diabetes.rwrite1.txt` | `age sex bmi map tc ldl hdl tch ltg glu y` |

Son el mismo dataset con distinto nombre de columna: lo que en la versión de R se llama
`map` (mean arterial pressure) acá se llama **`BP`**. Si mezclás los nombres, el `KeyError`
es seguro.

Sobre `SEX`: la página de la fuente **no documenta** qué código corresponde a cada sexo,
sólo dice "sex". Los valores observados son `1` y `2`. Por eso todo el análisis se hace
por grupo (`Sexo 1` / `Sexo 2`) sin asumir cuál es cuál: la conclusión estadística es
válida igual, pero no podemos etiquetar los grupos como hombre/mujer sin el codebook.

## 0. Entorno y carga

Dependencias: `pandas`, `numpy`, `matplotlib`, `seaborn` y `scipy` (para los tests
estadísticos), declaradas en `pyproject.toml` y sincronizadas con `uv sync`.

La primera celda imprime `sys.executable`. Si ahí aparece un Python distinto al `.venv` del
proyecto, el **kernel está mal elegido** y vas a ver `ModuleNotFoundError` aunque el
paquete esté instalado: hay que seleccionar el intérprete del `.venv` en el IDE.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
from scipy import stats

# Estilo de los gráficos
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

URL = "https://www4.stat.ncsu.edu/~boos/var.select/diabetes.tab.txt"
df = pd.read_csv(URL, sep="\t")

print(f"Intérprete: {sys.executable}")
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"scipy {scipy.__version__} | seaborn {sns.__version__} | matplotlib {plt.matplotlib.__version__}")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")

df.head()

In [ ]:
# SEX viene codificado 1 / 2. Mantenemos el código numérico original y agregamos
# una versión de texto sólo para etiquetas y agrupaciones.
df["SEXO"] = df["SEX"].map({1: "Sexo 1", 2: "Sexo 2"})

numericas = df.select_dtypes("number")  # todas menos SEXO
df.dtypes.to_frame("tipo")

## 1. Valores medios y varianza

`SEX` es una variable categórica codificada con números: calcularle media o varianza
"funciona" pero no significa nada. Por eso el resumen se hace sobre las variables
numéricas y se aclara aparte.

In [ ]:
resumen = pd.DataFrame(
    {
        "media": numericas.mean(),
        "varianza": numericas.var(),
        "desvio": numericas.std(),
        "minimo": numericas.min(),
        "maximo": numericas.max(),
        "sesgo": numericas.skew(),
    }
).round(3)

print(f"Conteo de sexo: {df['SEXO'].value_counts().to_dict()}")
resumen

Las gráficas de barras comparan variables medidas en **unidades distintas** (años,
kg/m², mmHg, unidades de laboratorio). Sirven para ver el orden de magnitud de cada una,
no para compararlas entre sí: la barra más alta es la que tiene la unidad más grande,
no la que "varía más". Para eso está la sección 4.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

resumen["media"].plot.bar(ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Valor medio por variable")
axes[0].set_ylabel("Media (en la unidad de cada variable)")
axes[0].set_xlabel("")

resumen["desvio"].plot.bar(ax=axes[1], color="darkorange", edgecolor="black")
axes[1].set_title("Desvío estándar por variable")
axes[1].set_ylabel("Desvío estándar (en la unidad de cada variable)")
axes[1].set_xlabel("")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 2. Boxplots de IMC, PA y Y según el género

`BP` es la presión arterial media (el `map` de la otra versión del archivo) y `Y` la
progresión de la enfermedad.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

graficos = [
    ("BMI", "IMC (BMI)"),
    ("BP", "Presión arterial media (BP)"),
    ("Y", "Progresión de la enfermedad (Y)"),
]

for ax, (columna, titulo) in zip(axes, graficos):
    sns.boxplot(
        data=df,
        x="SEXO",
        y=columna,
        hue="SEXO",
        palette="Set2",
        showmeans=True,
        legend=False,
        ax=ax,
    )
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel(columna)

plt.tight_layout()
plt.show()

En los tres casos las cajas se solapan casi por completo: a simple vista el sexo
aporta poco para separar a los pacientes. Los triángulos (medias) quedan por encima de las
medianas en `Y`, señal de una distribución asimétrica hacia la derecha. Eso se confirma en
la sección 3 y se pone a prueba formalmente en la sección 5.

## 3. Distribución de Edad, Sexo, IMC e Y

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.histplot(data=df, x="AGE", bins=20, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Edad (AGE)")

sns.countplot(data=df, x="SEXO", hue="SEXO", palette="Set2", legend=False, ax=axes[0, 1])
axes[0, 1].set_title("Sexo (SEX)")
axes[0, 1].set_xlabel("")

sns.histplot(data=df, x="BMI", bins=20, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Índice de masa corporal (BMI)")

sns.histplot(data=df, x="Y", bins=25, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("Progresión de la enfermedad (Y)")

plt.tight_layout()
plt.show()

In [ ]:
resumen[["media", "desvio", "minimo", "maximo", "sesgo"]].loc[["AGE", "SEX", "BMI", "Y"]]

Lectura:

* **Edad**: prácticamente simétrica (sesgo −0.23), centrada en 48.5 años, con casos de 19
  a 79. La muestra está sesgada hacia adultos mayores: hay pocos pacientes jóvenes.
* **Sexo**: muestra razonablemente balanceada (235 del grupo 1 y 207 del grupo 2), así que
  cualquier comparación entre grupos tiene potencia suficiente.
* **IMC**: concentrada entre 20 y 30, con sesgo leve a la derecha (0.60) y un máximo de
  42.2.
* **Y**: la variable con mayor dispersión relativa (desvío 77.1 sobre una media de 152.1).
  El 50% central va de 85 a 200, con una cola de casos severos que llega a 346.

La asimetría de `Y` (sesgo 0.44, media 152.1 > mediana 140.5) importa: justifica mirar la
mediana además de la media y justifica acompañar el test de medias con un test no
paramétrico.

## 4. Correlación entre variables y progresión de la enfermedad

Se calcula Pearson sobre todas las variables numéricas. Con 10 coeficientes contra `Y`
conviene corregir por comparaciones múltiples: si probás 10 correlaciones al 5%, esperás
alrededor de 0.5 falsos positivos. Se aplica la corrección de Benjamini-Hochberg (FDR).

In [ ]:
correlaciones = numericas.corr(method="pearson")["Y"].drop("Y")

tabla_corr = pd.DataFrame({"r": correlaciones})

tabla_corr["p_valor"] = [
    stats.pearsonr(numericas[col], numericas["Y"]).pvalue for col in tabla_corr.index
]
tabla_corr["p_ajustado_bh"] = stats.false_discovery_control(tabla_corr["p_valor"])
tabla_corr["significativa"] = tabla_corr["p_ajustado_bh"] < 0.05
tabla_corr["r2"] = tabla_corr["r"] ** 2

tabla_corr = tabla_corr.sort_values("r2", ascending=False).round(4)
tabla_corr

Atención: `SEX` entra en la tabla porque está codificado con números, pero es una
variable binaria. Su "correlación" sólo mide la diferencia de medias entre los dos grupos,
que es justamente lo que se prueba en la sección 5. Tomarla como una correlación más es una
licencia: el código de `SEX` es arbitrario, si lo invirtiéramos el signo de `r` cambiaría.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

matriz = numericas.corr(method="pearson")
sns.heatmap(
    matriz,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"shrink": 0.8},
    annot_kws={"size": 7},
    ax=axes[0],
)
axes[0].set_title("Matriz de correlación (Pearson)")

tabla_corr["r"].sort_values().plot.barh(
    ax=axes[1],
    color=["indianred" if r < 0 else "seagreen" for r in tabla_corr["r"].sort_values()],
    edgecolor="black",
)
axes[1].set_title("Correlación con Y")
axes[1].set_xlabel("r de Pearson")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

El heatmap muestra además dos pares de predictores con colinealidad fuerte: `S1`–`S2`
(r = 0.90) y `S3`–`S4` (r = −0.74). Eso no afecta la correlación con `Y`, pero sí sería un
problema para un modelo de regresión: aportan información redundante.

In [ ]:
# La relación más fuerte con Y, variable por variable, en un solo gráfico
fuertes = tabla_corr["r"].drop("SEX").abs().sort_values(ascending=False).index[:3]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, columna in zip(axes, fuertes):
    sns.regplot(
        data=df,
        x=columna,
        y="Y",
        ax=ax,
        scatter_kws={"alpha": 0.4, "s": 18},
        line_kws={"color": "black"},
    )
    ax.set_title(f"{columna} vs Y  (r = {tabla_corr.loc[columna, 'r']:.3f})")
    ax.set_xlabel(columna)
    ax.set_ylabel("Y")

plt.tight_layout()
plt.show()

`S3` es el único predictor con correlación negativa marcada: a mayor `S3`, menor
progresión. Es una variable derivada del colesterol, así que el signo no es interpretable
por sí solo sin el codebook clínico.

Lo importante del objetivo: **BMI, S5 y BP son los que mejor se asocian con Y**, y `AGE`
aporta poco (r ≈ 0.19, aunque con n = 442 alcanza significancia). Correlación no es
causalidad: acá sólo se mide asociación lineal.

## 5. ¿La progresión es distinta entre hombres y mujeres?

**H₀**: la progresión media es igual en los dos grupos de sexo.
**H₁**: la progresión media difiere.

Se usa un **t-test de Welch** (no asume varianzas iguales, que es lo correcto cuando no
tenemos motivo para suponerlas). Se reporta también la diferencia de medias con su
intervalo de confianza al 95% y el tamaño del efecto, porque un p-valor solo no dice
cuánto importa la diferencia.

Como `Y` es asimétrica, se agrega la **U de Mann-Whitney** como control: si ambas pruebas
coinciden, la conclusión no depende del supuesto de normalidad.

In [ ]:
grupo_1 = df.loc[df["SEX"] == 1, "Y"]
grupo_2 = df.loc[df["SEX"] == 2, "Y"]

# Estadísticos descriptivos por grupo
descriptivo = pd.DataFrame(
    {
        "Sexo 1": grupo_1.describe(),
        "Sexo 2": grupo_2.describe(),
    }
).round(2)
print("Descriptivos de Y por grupo")
print(descriptivo)

# t-test de Welch (equal_var=False) + intervalo de confianza de la diferencia
welch = stats.ttest_ind(grupo_1, grupo_2, equal_var=False)
ic_baja, ic_alta = welch.confidence_interval(confidence_level=0.95)

# Test no paramétrico
mann = stats.mannwhitneyu(grupo_1, grupo_2, alternative="two-sided")

# Tamaño del efecto: d de Cohen con desvío agrupado
n1, n2 = len(grupo_1), len(grupo_2)
s_agrupado = np.sqrt(
    ((n1 - 1) * grupo_1.var(ddof=1) + (n2 - 1) * grupo_2.var(ddof=1)) / (n1 + n2 - 2)
)
d_cohen = (grupo_1.mean() - grupo_2.mean()) / s_agrupado


def interpretar_d(d: float) -> str:
    d = abs(d)
    if d < 0.2:
        return "despreciable"
    if d < 0.5:
        return "pequeño"
    if d < 0.8:
        return "mediano"
    return "grande"


resultado = pd.DataFrame(
    [
        {
            "prueba": "t de Welch",
            "estadistico": round(float(welch.statistic), 4),
            "p_valor": round(float(welch.pvalue), 4),
        },
        {
            "prueba": "U de Mann-Whitney",
            "estadistico": round(float(mann.statistic), 1),
            "p_valor": round(float(mann.pvalue), 4),
        },
    ]
)

print(f"Diferencia de medias (Sexo 1 - Sexo 2): {grupo_1.mean() - grupo_2.mean():.2f} puntos")
print(f"IC 95% de la diferencia: [{ic_baja:.2f}, {ic_alta:.2f}]")
print(f"d de Cohen: {d_cohen:.3f} (efecto {interpretar_d(d_cohen)})")
print(resultado)

if welch.pvalue < 0.05:
    print("Con alpha = 0.05 se rechaza H0: hay evidencia de diferencia entre los grupos.")
else:
    print("Con alpha = 0.05 NO se rechaza H0: no hay evidencia de diferencia entre los grupos.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df, x="SEXO", y="Y", hue="SEXO", palette="Set2", showmeans=True, legend=False, ax=axes[0]
)
axes[0].set_title("Progresión (Y) por grupo de sexo")
axes[0].set_xlabel("")
axes[0].set_ylabel("Y")

sns.kdeplot(data=df, x="Y", hue="SEXO", fill=True, common_norm=False, alpha=0.4, ax=axes[1])
axes[1].set_title("Densidad de Y por grupo de sexo")
axes[1].set_xlabel("Y")
axes[1].set_ylabel("Densidad")

plt.tight_layout()
plt.show()

Las dos curvas de densidad quedan una encima de la otra: la diferencia entre grupos es
mucho menor que la variabilidad dentro de cada grupo. Que el p-valor no sea significativo
no significa "los grupos son iguales", significa que con estos datos no hay evidencia
suficiente para afirmar que difieren.

## Conclusiones

1. **Descriptivos**: `Y` es la variable con mayor dispersión relativa (CV 0.51) y presenta
   asimetría moderada a la derecha (media 152.1 vs mediana 140.5). `SEX` es categórica y no
   se resume con media.
2. **Boxplots por sexo**: las tres variables se solapan casi completamente entre grupos.
3. **Distribuciones**: `AGE` es casi simétrica, `BMI` tiene sesgo leve a la derecha (0.60),
   `SEX` está balanceada y `Y` tiene una cola derecha moderada.
4. **Correlación con Y**: `BMI` (r = 0.586) y `S5` (r = 0.566) son las asociaciones más
   fuertes; `BP` (0.441) y `S4` (0.430) siguen; `S3` es la única negativa fuerte (−0.395).
   `AGE` y los lípidos plasmáticos (`S1`, `S2`) aportan poco.
5. **Hipótesis por sexo**: no se rechaza H₀ (Welch p > 0.05). Con 442 pacientes y una
   diferencia de medias de sólo ~6.7 puntos sobre un desvío de ~77, el tamaño del efecto es
   despreciable: el sexo no explica la progresión en esta muestra.

**Limitación principal**: el código de `SEX` no está documentado en la fuente, así que las
conclusiones se enuncian por grupo y no por "hombres" y "mujeres".